In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

## Generate Subgroup Classes

In [66]:
df = pd.read_csv('/Users/yuw/Documents/Development/Colon_Data/data/coad+uk.csv')
out_path = '/Users/yuw/Documents/Development/Colon_Data/data/coad+uk.csv'

In [67]:
df[['tumor%', 'lymph%', 'til%', 'tumorSize']].describe()

,tumor%,lymph%,til%,tumorSize
count,58.000000,58.000000,58.000000,58.000000
mean,0.314895,0.133186,0.287097,5.753448
std,0.141661,0.062267,0.141216,3.622939
min,0.041400,0.020900,0.028500,1.200000
25%,0.209650,0.081850,0.170050,3.850000
50%,0.299050,0.125450,0.289200,5.000000
75%,0.392125,0.174850,0.399125,6.975000
max,0.706200,0.256200,0.627300,21.000000


In [ ]:
df['OS.time']=df['OS.time'].replace(np.nan,round(df['OS.time'].mean()))
df['PFI.time']=df['PFI.time'].replace(np.nan,round(df['PFI.time'].mean()))

In [6]:
df.loc[df.ajcc_stage == 'Stage IIA', 'stage'] = 'Stage II'
df.loc[df.ajcc_stage == 'Stage I', 'stage'] = 'Stage I'
df.loc[df.ajcc_stage == 'Stage IIIB', 'stage'] = 'Stage III'
df.loc[df.ajcc_stage == 'Stage IV', 'stage'] = 'Stage IV'
df.loc[df.ajcc_stage == 'Stage IIIC', 'stage'] = 'Stage III'
df.loc[df.ajcc_stage == 'Stage II', 'stage'] = 'Stage II'
df.loc[df.ajcc_stage == 'Stage IVA', 'stage'] = 'Stage IV'
df.loc[df.ajcc_stage == 'Stage III', 'stage'] = 'Stage III'
df.loc[df.ajcc_stage == 'Stage IIIA', 'stage'] = 'Stage III'
df.loc[df.ajcc_stage == 'Stage IIB', 'stage'] = 'Stage II'
df.loc[df.ajcc_stage == 'Stage IIC', 'stage'] = 'Stage II'
df.loc[df.ajcc_stage == 'Stage IVB', 'stage'] = 'Stage IV'
df.loc[df.ajcc_stage == 'Stage IA', 'stage'] = 'Stage I'
df.loc[df.ajcc_stage == '[Not Available]', 'stage'] = '[Not Available]'
df.loc[df.ajcc_stage == '[Discrepancy]', 'stage'] = '[Discrepancy]'

In [30]:
# tumor budding score - UKentucky*
df.loc[(df['buddingScore']=='G1')|(df['buddingScore']=='G2'), 'buddingScore_status']='low'
df.loc[df['buddingScore']=='G3', 'buddingScore_status']='high'

### Classified by median value

In [54]:
# threshold = round(df[label].quantile(0.5), 3)
threshold = 0.126
def SetSubAJCCgroups(label, gpname_h, gpname_l):
    # Tumor, Lymph, and TILs high vs low with AJCC status
    df.loc[(df['AJCC_stage']=='Stages III-IV')&(df[label]>threshold),gpname_h]='high'
    df.loc[(df['AJCC_stage']=='Stages III-IV')&(df[label]<=threshold),gpname_h]='low'
    df.loc[(df['AJCC_stage']=='Stages I-II')&(df[label]>threshold),gpname_l]='high'
    df.loc[(df['AJCC_stage']=='Stages I-II')&(df[label]<=threshold),gpname_l]='low'    
    
def SetSubMSIgroups(label, gpname_h, gpname_l):
    # Tumor, Lymph, and TILs high vs low with MSI status
    df.loc[(df['MSI']=='MSS/MSI-L')&(df[label]>threshold), gpname_h] = 'high'
    df.loc[(df['MSI']=='MSS/MSI-L')&(df[label]<=threshold), gpname_h] = 'low'
    df.loc[(df['MSI']=='MSI-H')&(df[label]>threshold), gpname_l] = 'high'
    df.loc[(df['MSI']=='MSI-H')&(df[label]<=threshold), gpname_l] = 'low'

def SetSubTumorBuddingGroups(label, gpname_h, gpname_l):
    df.loc[(df['buddingScore_status']=='Bd3')&(df[label]>threshold), gpname_h]='high'
    df.loc[(df['buddingScore_status']=='Bd3')&(df[label]<=threshold), gpname_h]='low'
    df.loc[(df['buddingScore_status']=='Bd1/Bd2')&(df[label]>threshold), gpname_l]='high'
    df.loc[(df['buddingScore_status']=='Bd1/Bd2')&(df[label]<=threshold), gpname_l]='low'
        
def SetSubgroups(label, gpname):
    # Tumor, Lymph, TILs high vs low
    df.loc[df[label]>threshold, gpname] = 'high'
    df.loc[df[label]<=threshold, gpname] = 'low'

def SetSizegroups(label, gpname):
    # Tumor, Lymph, TILs high vs low
    df.loc[df[label]>round(df['tumorSize'].median(), 3), gpname] = 'high'
    df.loc[df[label]<=round(df['tumorSize'].median(), 3), gpname] = 'low'    

def SetTumorSizeSubgroups(label, gpname_h, gpname_l):
    # Lymph or TILs high vs low with tumor_status

    df.loc[(df['tumorSize']>round(df['tumorSize'].median(),3))&(df[label]>threshold), gpname_h] = 'high'
    df.loc[(df['tumorSize']>round(df['tumorSize'].median(),3))&(df[label]<=threshold),gpname_h] = 'low'
    df.loc[(df['tumorSize']<=round(df['tumorSize'].median(),3))&(df[label]>threshold), gpname_l] = 'high'
    df.loc[(df['tumorSize']<=round(df['tumorSize'].median(),3))&(df[label]<=threshold), gpname_l] = 'low'
    
def SetAssociateSubgroups(label_0, label_1, gpname_h, gpname_l):
    # Lymph or TILs high vs low with tumor_status
    df.loc[(df[label_0]>round(df[label_0].median(),3))&(df[label_1]>threshold), gpname_h] = 'high'
    df.loc[(df[label_0]>round(df[label_0].median(),3))&(df[label_1]<=threshold),gpname_h] = 'low'
    df.loc[(df[label_0]<=round(df[label_0].median(),3))&(df[label_1]>threshold), gpname_l] = 'high'
    df.loc[(df[label_0]<=round(df[label_0].median(),3))&(df[label_1]<=threshold), gpname_l] = 'low'

### TCGA COAD

In [64]:
# Main groups
# SetSizegroups('tumorSize', 'tumorSize_status')
# SetSubgroups('til%', 'til_status')

# # AJCC
# SetSubAJCCgroups('til%', 'hajcc_til', 'lajcc_til')
# SetSubgroups('tumor%', 'tumorCon_status')
# # Tumor Size
# SetTumorSizeSubgroups('til%', 'lsize_til', 'hsize_til')

# MSI
SetSubMSIgroups('til%', 'lmsi_til', 'hmsi_til')

# Tumor content and Tumor size associate with TILs
# SetAssociateSubgroups('tumor%', 'til%', 'htumor_til', 'ltumor_til')

In [65]:
df.to_csv(out_path, index=False)

### UKentucky

In [68]:
# Main groups
SetSubgroups('til%', 'til_status')

# AJCC
SetSubAJCCgroups('til%', 'hajcc_til', 'lajcc_til')

# MSI
# SetSubMSIgroups('til%', 'hmsi_til', 'lmsi_til')

# Tumor content and Tumor size associate with TILs
SetTumorSizeSubgroups('til%', 'hsize_til', 'lsize_til')

# Tumor budding scores - UKentucky*
SetSubTumorBuddingGroups('til%', 'hscore_til', 'lscore_til')

SetAssociateSubgroups('tumor%','til%', 'htumor_til', 'ltumor_til')

In [69]:
df.to_csv(out_path, index=False)

In [37]:
df['hajcc_til'].value_counts()

hajcc_til
high    20
low      5
Name: count, dtype: int64